![servicedesk](servicedesk.png)

CleverSupport is a company at the forefront of AI innovation, specializing in the development of AI-driven solutions to enhance customer support services. Their latest endeavor is to engineer a text classification system that can automatically categorize customer complaints. 

Your role as a data scientist involves the creation of a sophisticated machine learning model that can accurately assign complaints to specific categories, such as mortgage, credit card, money transfers, debt collection, etc.

In [ ]:
pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1

In [ ]:
pip uninstall -y torch torchvision torchaudio torchtext torchmetrics torchdata torcheval torchsummary torchinfo torch-tb-profiler


In [ ]:
pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1


In [ ]:
pip install torchmetrics==1.3.2


In [1]:
pip list | grep torch


torch                                    2.7.1
torchaudio                               2.7.1
torchmetrics                             1.3.2
torchvision                              0.22.1
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
import nltk, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
#from torchmetrics import Accuracy, Precision, Recall

In [3]:
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall

/Users/phuongnguyen/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/phuongnguyen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
# Import data and labels
with open("words.json", 'r') as f1:
    words = json.load(f1)
with open("text.json", 'r') as f2:
    text = json.load(f2)
labels = np.load('labels.npy')

In [6]:
# Dictionaries to store the word to index mappings and vice versa
word2idx = {o:i for i,o in enumerate(words)}
idx2word = {i:o for i,o in enumerate(words)}

# Looking up the mapping dictionary and assigning the index to the respective words
for i, sentence in enumerate(text):
    text[i] = [word2idx[word] if word in word2idx else 0 for word in sentence]
    
# Defining a function that either shortens sentences or pads sentences with 0 to a fixed length
def pad_input(sentences, seq_len):
    features = np.zeros((len(sentences), seq_len),dtype=int)
    for ii, review in enumerate(sentences):
        if len(review) != 0:
            features[ii, -len(review):] = np.array(review)[:seq_len]
    return features

text = pad_input(text, 50)

In [7]:
print(word2idx)
print(idx2word)

{'_PAD': 0, '_UNK': 1, 'the': 2, '.': 3, 'i': 4, 'to': 5, ',': 6, 'and': 7, 'a': 8, 'my': 9, 'that': 10, 'of': 11, 'was': 12, 'in': 13, 'on': 14, 'they': 15, 'for': 16, 'me': 17, 'not': 18, 'this': 19, 'have': 20, 'is': 21, 'with': 22, 'account': 23, 'it': 24, '$': 25, 'from': 26, '{': 27, '}': 28, 'credit': 29, 'bank': 30, 'as': 31, 'had': 32, 'be': 33, ')': 34, '(': 35, '``': 36, 'would': 37, 'money': 38, 'an': 39, 'by': 40, 'at': 41, 'we': 42, 'payment': 43, 'but': 44, 'them': 45, 'or': 46, 'no': 47, 'been': 48, 'told': 49, 'did': 50, 'has': 51, 'card': 52, 'are': 53, 'received': 54, 'do': 55, 'their': 56, 'when': 57, 'were': 58, 'all': 59, 'which': 60, 'you': 61, 'called': 62, 'so': 63, 'loan': 64, "n't": 65, 'any': 66, 'am': 67, 'after': 68, 'back': 69, 'if': 70, 'time': 71, 'mortgage': 72, 'he': 73, 'she': 74, 'information': 75, 'out': 76, 'sent': 77, 'because': 78, 'company': 79, 'will': 80, 'debt': 81, "'s": 82, 'then': 83, 'never': 84, 'pay': 85, 'get': 86, 'can': 87, 'there':

In [8]:
print(text[:2])

[[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    4   62   78    4   20   48  476  779    5  953
   215    8  149   26   45  316    8   81    7    2  230   62   17    8
  5025   68    4  103   95 6772    9   23]
 [  19   89  276  445   41  441 3766    6    4   32  577   10   88   58
   217 2952 1608   14    9   29  118    3    4   62 1255  927    5  312
    76  154    4   32  275 7917   10   58   18   88  191    3   68    8
   382 5471    6    4   12 7918   22 1325]]


In [9]:
# Splitting dataset
train_text, test_text, train_label, test_label = train_test_split(text, labels, test_size=0.2, random_state=42)

train_data = TensorDataset(torch.from_numpy(train_text), torch.from_numpy(train_label).long())
test_data = TensorDataset(torch.from_numpy(test_text), torch.from_numpy(test_label).long())

In [35]:
train_loader = DataLoader(train_data, shuffle=True, batch_size=32)
test_loader = DataLoader(test_data, shuffle=False, batch_size=32)

In [10]:
unique_count = len(set(train_label))
print(unique_count)

5


In [20]:
print(train_data[0])

(tensor([   4,   32,    8,  162,  190,   23,   41,  124,   30,    6,   60,  117,
         301,   51,   48,  198,    7,  909,    5,   33,  198,   90,    5,  275,
           8,  149,    5, 1847,   17, 3695,    6, 1646,    6,    7, 1866,    6,
           5,    2,  384,   11, 4383,   11, 1458, 1208, 2724, 3029,   26,    2,
        1045, 1013]), tensor(0))


In train_label dataset, there are 5 classes

**1. Define the classifier**

Define a class containing all the appropriate layers, and a method to perform the forward pass over a batch of input text.
* Creating a class to contain the layers of the classifier
    * Use PyTorch's nn.Embedding class to define the embedding layer.
    * Create an instance of it in the TicketClassifier class's constructor and assign it to an instance variable such as self.embedding.
* Adding an embedding layer
    * Use PyTorch's nn.Embedding class to define the embedding layer.
    * Create an instance of it in the TicketClassifier class's constructor and assign it to an instance variable such as self.embedding.
* Adding a convolution ayer
    * Use PyTorch's nn.Conv1d class to define the 1D convolution layer.
    * Create an instance of it in the TicketClassifier class's constructor and assign it to an instance variable such as self.conv.
* Adding a linear layer
    * Use PyTorch's nn.Linear class to define the linear layer.
    * Create an instance of it in the TicketClassifier class's constructor and assign it to an instance variable such as self.fc.
* Define a .forward() method
    * Finally, define a .forward() method that passes the input through the embedding and convolution layer, applies nn.functional.relu on the output, and finally applies linear layer before returning the output.

Note: 
- conv.mean(dim=2) (batch_size, embed_dim, seq_len) -> (batch_size, embed_dim) để biến đổi vector cố định chiều dài, không phụ thuộc vào seq_len để đưa vào nn.Linear
mean(dim=2) nghĩa là lấy trung bình theo chiều sequence (trung bình tất các các time-step/token features) => Global Average Pooling (GAP)
Tương tự như Fully-connected mọi pixel, sử dụng GAP để giảm chiều.

In [11]:
# Text classification model using CNN
class SentimentAnalysisCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size = 3, stride=1, padding=1)
        self.fc = nn.Linear(embed_dim, 5) # 5 classes
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)    # (batch_size, seq_len, embed_dim) -> (batch_size, embed_dim, seq_len)
        conved = F.relu(self.conv(embedded))                # (batch_size, embed_dim, seq_len)
        conved = conved.mean(dim=2)                         # (batch_size, embed_dim)
        return self.fc(conved)

**#2.Training the classifier**

Define a training loop that loops over the dataset, calculating the loss and propagating it backwards through the network.
* Define a suitable loss criterion
    * Use PyTorch's nn.CrossEntropyLoss, since this is a multi-class classification problem.
* Define an optimizer
    * Use PyTorch's optim.Adam optimizer.

In [12]:
vocab_size = len(word2idx)
embedding_dim = 300

In [13]:
model = SentimentAnalysisCNN(vocab_size, embedding_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [25]:
#Test data
batch_size = 32
input_test = train_data[:batch_size]
print(input_test)

(tensor([[   4,   32,    8,  ...,    2, 1045, 1013],
        [  14,    6,  302,  ...,  216,  243,  365],
        [  41,  156,    4,  ..., 2211,   10,   15],
        ...,
        [  14,   42,   58,  ...,   27,   25, 3882],
        [ 187,  186,  296,  ...,   11,  156,   27],
        [   2,   30,   11,  ...,    2,  742,   96]]), tensor([0, 0, 2, 4, 0, 3, 0, 0, 3, 3, 4, 1, 4, 4, 3, 3, 2, 0, 4, 0, 0, 0, 2, 0,
        1, 1, 3, 0, 0, 1, 1, 0]))


In [29]:
output_test = model(input_test[0])
print(output_test.shape)

torch.Size([32, 5])


In [31]:
print(input_test[1].shape)

torch.Size([32])


In [32]:
loss = criterion(output_test, input_test[1])

In [30]:
#Test probabilities
test_probs = F.softmax(output_test, dim=-1)
test_preds = torch.argmax(test_probs, dim=-1)
print("Probabilities:", test_probs)
print("Predicted classes:", test_preds)

Probabilities: tensor([[0.2023, 0.1805, 0.1957, 0.1981, 0.2234],
        [0.2095, 0.1792, 0.1852, 0.1860, 0.2401],
        [0.2172, 0.1762, 0.1910, 0.1849, 0.2307],
        [0.2053, 0.1734, 0.1939, 0.1817, 0.2457],
        [0.2042, 0.1811, 0.1876, 0.1922, 0.2349],
        [0.2067, 0.1777, 0.1928, 0.1990, 0.2238],
        [0.2032, 0.1828, 0.1821, 0.1942, 0.2377],
        [0.2103, 0.1847, 0.1954, 0.1777, 0.2319],
        [0.2054, 0.1802, 0.1922, 0.1969, 0.2253],
        [0.1932, 0.1850, 0.1810, 0.2007, 0.2402],
        [0.2018, 0.1825, 0.1957, 0.1864, 0.2337],
        [0.1726, 0.1850, 0.1738, 0.2050, 0.2636],
        [0.1966, 0.1785, 0.1904, 0.1998, 0.2347],
        [0.2035, 0.1784, 0.1939, 0.1883, 0.2359],
        [0.2018, 0.1789, 0.1990, 0.1852, 0.2352],
        [0.2019, 0.1790, 0.1892, 0.1896, 0.2403],
        [0.1964, 0.1849, 0.1857, 0.1878, 0.2452],
        [0.2025, 0.1818, 0.1954, 0.1872, 0.2332],
        [0.2068, 0.1730, 0.1881, 0.1905, 0.2415],
        [0.2092, 0.1884, 0.1922, 0.

In [34]:
for inputs, labels in train_data:
    print(inputs)
    print(inputs.shape)
    print(labels)
    print(labels.shape)
    break

tensor([   4,   32,    8,  162,  190,   23,   41,  124,   30,    6,   60,  117,
         301,   51,   48,  198,    7,  909,    5,   33,  198,   90,    5,  275,
           8,  149,    5, 1847,   17, 3695,    6, 1646,    6,    7, 1866,    6,
           5,    2,  384,   11, 4383,   11, 1458, 1208, 2724, 3029,   26,    2,
        1045, 1013])
torch.Size([50])
tensor(0)
torch.Size([])


In [36]:
#Training epoch
for epoch in range(10):
    model.train()
    training_loss = 0.0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    epoch_loss = training_loss / len(train_data)
    print(f"Epoch {epoch+1}, Training Loss: {epoch_loss:.4f}")

Epoch 1, Training Loss: 0.0334
Epoch 2, Training Loss: 0.0189
Epoch 3, Training Loss: 0.0142
Epoch 4, Training Loss: 0.0108
Epoch 5, Training Loss: 0.0079
Epoch 6, Training Loss: 0.0054
Epoch 7, Training Loss: 0.0036
Epoch 8, Training Loss: 0.0024
Epoch 9, Training Loss: 0.0015
Epoch 10, Training Loss: 0.0009


**#3. Testing the classifier**

Use your trained model to classify the text in the test set, and calculate the appropriate metrics.
* Predict the category of each ticket in the test data.
    * Invoke model() on your input data to pass the data through the network.
    * Use torch.argmax() to find the category with the highest predicted probability.
* Calculate the accuracy
    * Use torchmetrics.Accuracy to calculate the accuracy.
* Calculate the precision and recall
    * Use torchmetrics.Precision and torchmetrics.Recall to calculate the precision and recall.

In [37]:
# Valiation  
validation_loss = 0.0
model.eval()

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        validation_loss += loss.item()
epoch_val_loss = validation_loss / len(test_loader)
model.train()


SentimentAnalysisCNN(
  (embedding): Embedding(10146, 300)
  (conv): Conv1d(300, 300, kernel_size=(3,), stride=(1,), padding=(1,))
  (fc): Linear(in_features=300, out_features=5, bias=True)
)

In [38]:
print(f"Validation Loss: {epoch_val_loss:.4f}")

Validation Loss: 0.8139


In [40]:
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall

In [42]:
#Compute metric
accuracy = MulticlassAccuracy( num_classes=5).to("cpu")
precision = MulticlassPrecision(num_classes=5, average="macro").to("cpu")
recall = MulticlassRecall(num_classes=5, average="macro").to("cpu")

In [44]:
#Evaluation metrics
model.eval()
all_acc, all_prec, all_rec = 0, 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_acc += accuracy(preds, labels).item()
        all_prec += precision(preds, labels).item()
        all_rec += recall(preds, labels).item()
    
#Compute metrics
acc = accuracy.compute().item()
prec = precision.compute().item()
rec = recall.compute().item()
print(f"Epoch {epoch+1}: Loss={epoch_loss:.4f}, Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}")

#Reset metric for next epoch
accuracy.reset()
precision.reset()
recall.reset()

Epoch 10: Loss=0.0009, Acc=0.7937, Prec=0.7969, Rec=0.7937
